# Patching d'activation incrémental

Démontre le coût O(|V_θ|) du patching d'activation dans NeuroDSL : patcher un nœud interne n'invalide que son cône en aval (`_invalidate_downstream!` + `demand!`), contrairement à PyTorch/TransformerLens où chaque patch coûte un forward complet.

Modèle de test : `LlamaModel` (déjà exporté et testé dans le package), alimenté directement par une matrice aléatoire `(seq_len, dim)` -- pas d'embeddings ni de tête LM, on démontre une capacité système, pas un résultat de langage.

Protocole : un run "propre" et un run "corrompu" (un seul token remplacé), sur le même graphe -- le passage propre → corrompu invalide tout le graphe via `set!(g,:input,...)`, ce qui donne gratuitement le coût "forward complet" de référence (l'équivalent du coût PyTorch/TransformerLens, mesuré avec le même moteur).

In [1]:
using NeuroDSL, Statistics, Random, Printf, LinearAlgebra

dev = NeuroDSL.Backend.CPUDevice()
dim, n_heads, hidden_dim, n_layers, seq_len = 128, 8, 512, 8, 16
ns = :patch_bench

g = NeuroDSL.NeuroGraph(namespace=ns, device=dev)
Random.seed!(42)
NeuroDSL.set!(g, :input, randn(Float32, seq_len, dim); namespace=ns)
output_sym = NeuroDSL.LlamaModel(n_layers, dim, n_heads, hidden_dim)(g, :input; namespace=ns)
println("Modèle : $(n_layers) couches LlamaBlock, dim=$(dim), n_heads=$(n_heads), seq_len=$(seq_len)")
println("Nœud de sortie : :$(output_sym)")

Modèle : 8 couches LlamaBlock, dim=128, n_heads=8, seq_len=16
Nœud de sortie : :layer_8_out


## Run propre et run corrompu

In [2]:
X_clean = randn(Float32, seq_len, dim)
NeuroDSL.set!(g, :input, X_clean; namespace=ns)
clean_output = copy(NeuroDSL.demand!(g, output_sym; namespace=ns))
clean_cache = NeuroDSL.capture_activations(g, ns)

X_corrupted = copy(X_clean)
X_corrupted[1, :] .= randn(Float32, dim)   # un seul token (position 1) corrompu

# Passage propre -> corrompu : set!(:input) invalide TOUT le graphe -- c'est
# exactement le coût d'un forward complet, mesuré ici comme référence gratuite.
t0 = time_ns()
NeuroDSL.set!(g, :input, X_corrupted; namespace=ns)
corrupted_output = copy(NeuroDSL.demand!(g, output_sym; namespace=ns))
full_forward_ms = (time_ns() - t0) / 1e6
corrupted_cache = NeuroDSL.capture_activations(g, ns)

@printf "Coût forward complet (référence, ≈ coût PyTorch/TransformerLens par patch) : %.3f ms\n" full_forward_ms

Coût forward complet (référence, ≈ coût PyTorch/TransformerLens par patch) : 24.128 ms


## Correction d'abord : deux invariants indépendants, avant toute mesure de vitesse

1. Le nœud patché doit garder sa valeur imposée après un `demand!` -- `set!` marquerait le nœud LUI-MÊME invalide (pas seulement ses successeurs), donc un `demand!` ultérieur le recalculerait silencieusement à partir de sa propre règle (toujours corrompue) et écraserait la valeur propre injectée. C'est exactement le bug détecté et corrigé pendant le développement de `patch_node!`.
2. Patcher la dernière couche (`layer_{n}_out`) doit être rigoureusement équivalent à patcher la sortie elle-même (`recovery == 1.0`) -- identité mathématique indépendante de toute seconde implémentation de référence, donc immunisée contre un bug partagé entre "implémentation" et "vérification".

In [3]:
patch_sym_mid = Symbol(:layer_, n_layers ÷ 2, :_out)
NeuroDSL.patch_node!(g, patch_sym_mid, clean_cache; namespace=ns)
right_after_patch = copy(NeuroDSL.node(g, patch_sym_mid; namespace=ns).value)
NeuroDSL.demand!(g, output_sym; namespace=ns)
still_after_demand = NeuroDSL.node(g, patch_sym_mid; namespace=ns).value
err = maximum(abs.(Array(right_after_patch) .- Array(still_after_demand)))
@printf "1) max|err| valeur patchée avant/après demand! : %.6e  %s\n" err (err < 1f-6 ? "✅" : "❌ ÉCHEC")

last_layer_sym = Symbol(:layer_, n_layers, :_out)
@printf "   layer_%d_out == output_sym ? %s\n" n_layers (last_layer_sym == output_sym ? "✅" : "❌")
NeuroDSL.patch_node!(g, patch_sym_mid, corrupted_cache; namespace=ns)  # restaurer avant la suite
NeuroDSL.demand!(g, output_sym; namespace=ns)
result_last = NeuroDSL.patch_and_measure!(g, output_sym, last_layer_sym, clean_cache, corrupted_cache,
                                           clean_output, corrupted_output; namespace=ns)
@printf "2) recovery en patchant la dernière couche (doit être 1.0) : %.6f  %s\n" result_last.recovery (isapprox(result_last.recovery, 1.0; atol=1e-5) ? "✅" : "❌ ÉCHEC")

@assert err < 1f-6 && isapprox(result_last.recovery, 1.0; atol=1e-5) "Correction non validée -- arrêt avant le benchmark de vitesse."
println("\n✅ Les deux invariants de correction sont validés.")

1) max|err| valeur patchée avant/après demand! : 0.000000e+00  ✅
   layer_8_out == output_sym ? ✅
2) recovery en patchant la dernière couche (doit être 1.0) : 1.000000  ✅

✅ Les deux invariants de correction sont validés.


## Benchmark : coût et récupération en fonction de la profondeur

Patcher **tout** le tableau d'une couche (toutes les positions à la fois) restaure trivialement 100% de la sortie, quelle que soit la profondeur -- par pur déterminisme, rejouer le calcul propre à partir de là reproduit forcément la sortie propre. Ça donne une courbe de *coût* valide, mais aucune courbe de *récupération* informative (vérifié empiriquement pendant le développement : `recovery == 1.000` pour les 8 couches sans distinction).

Pour une courbe de récupération qui montre où l'information du token corrompu se dilue dans le réseau, on ne patche que **la position du token corrompu** (ligne 1) à chaque couche, en laissant les autres positions telles que le run corrompu les a calculées -- protocole standard de causal tracing par position.

In [4]:
function trimmed_stats(v; frac=0.1)
    s = sort(v)
    k = round(Int, length(s)*frac)
    t = s[k+1:end-k]
    return mean(t), std(t)
end

function position_patch_cache(base_cache, patch_sym, clean_cache, row::Int)
    hybrid = copy(base_cache[patch_sym])
    hybrid[row, :] .= clean_cache[patch_sym][row, :]
    return Dict(patch_sym => hybrid)
end

println("Warmup...")
warm_sym = Symbol(:layer_, 1, :_out)
for _ in 1:3
    NeuroDSL.patch_and_measure!(g, output_sym, warm_sym,
                                 position_patch_cache(corrupted_cache, warm_sym, clean_cache, 1),
                                 corrupted_cache, clean_output, corrupted_output; namespace=ns)
end

println("\nCouche  |  temps médian (ms)  |  écart-type  |  recovery (position corrompue seule)")
println("-"^75)
results = NamedTuple[]
for i in 1:n_layers
    patch_sym = Symbol(:layer_, i, :_out)
    hybrid_cache = position_patch_cache(corrupted_cache, patch_sym, clean_cache, 1)
    times = Float64[]
    recov = 0.0
    for _ in 1:15
        r = NeuroDSL.patch_and_measure!(g, output_sym, patch_sym, hybrid_cache, corrupted_cache,
                                         clean_output, corrupted_output; namespace=ns)
        push!(times, r.time_ms)
        recov = r.recovery
    end
    m, sd = trimmed_stats(times)
    @printf "layer_%d  |  %8.3f ms        |  ±%.3f     |  %.3f\n" i m sd recov
    push!(results, (; layer=i, time_ms=m, std_ms=sd, recovery=recov))
end

@printf "\nRéférence forward complet : %.3f ms\n" full_forward_ms
println("\nRatio coût_patch / coût_forward_complet par couche :")
for r in results
    @printf "  layer_%d : %.1f%%\n" r.layer (r.time_ms / full_forward_ms * 100)
end

Warmup...

Couche  |  temps médian (ms)  |  écart-type  |  recovery (position corrompue seule)
---------------------------------------------------------------------------
layer_1  |     5.818 ms        |  ±0.430     |  0.753
layer_2  |     4.920 ms        |  ±0.291     |  0.666
layer_3  |     3.993 ms        |  ±0.173     |  0.613
layer_4  |     3.397 ms        |  ±0.209     |  0.565
layer_5  |     2.413 ms        |  ±0.230     |  0.521
layer_6  |     1.694 ms        |  ±0.138     |  0.461
layer_7  |     0.762 ms        |  ±0.051     |  0.435
layer_8  |     0.000 ms        |  ±0.000     |  0.386

Référence forward complet : 24.128 ms

Ratio coût_patch / coût_forward_complet par couche :
  layer_1 : 24.1%
  layer_2 : 20.4%
  layer_3 : 16.5%
  layer_4 : 14.1%
  layer_5 : 10.0%
  layer_6 : 7.0%
  layer_7 : 3.2%
  layer_8 : 0.0%


In [5]:
using Plots
gr()

layers = [r.layer for r in results]
cost_pct = [r.time_ms / full_forward_ms * 100 for r in results]
recovery_vals = [r.recovery for r in results]

mkpath("../figures")

plot(layers, cost_pct,
    marker = :circle, lw = 2, color = :steelblue, legend = false,
    xlabel = "Patched layer (depth)", ylabel = "Patch cost (% of full forward)",
    title = "Cost vs. depth")
savefig("../figures/patching_cost.pdf")

plot(layers, recovery_vals,
    marker = :circle, lw = 2, color = :firebrick, legend = false,
    xlabel = "Patched layer (depth)", ylabel = "Recovery (corrupted token only)",
    title = "Recovery vs. depth", ylim = (0, 1))
savefig("../figures/patching_recovery.pdf")

println("✅ figures/patching_cost.pdf and figures/patching_recovery.pdf saved")

✅ figures/patching_cost.pdf and figures/patching_recovery.pdf saved


## Balayage multi-sites amorti

Un balayage de causal tracing ne teste jamais un seul site : il en teste un par couche. `patch_and_measure!` restaure l'état corrompu en rappelant `patch_node!`+`demand!`, ce qui **recalcule** le même cône qu'à l'étape de mesure -- sur un balayage complet, ce recalcul de restauration coûte le même ordre de grandeur que les patches eux-mêmes.

La restauration n'a pourtant besoin d'aucun recalcul : le run corrompu de référence est déjà entièrement mis en cache (`corrupted_cache`, capturé une fois avant le premier patch). `restore_from_cache!` remplace donc le recalcul par une copie directe des valeurs déjà connues, et `sweep_patch_sites!` orchestre un balayage complet avec cette restauration rapide.

In [6]:
# ── Correction : restauration par cache == restauration par recalcul ──────
patch_sym_check = sites_layers = Symbol(:layer_, 2, :_out)
affected = NeuroDSL._downstream_nodes(g, patch_sym_check, ns)
NeuroDSL.patch_node!(g, patch_sym_check, clean_cache; namespace=ns)
NeuroDSL.demand!(g, output_sym; namespace=ns)
NeuroDSL.restore_from_cache!(g, ns, corrupted_cache, affected)
state_cache_restore = NeuroDSL.capture_activations(g, ns)

NeuroDSL.patch_node!(g, patch_sym_check, clean_cache; namespace=ns)
NeuroDSL.demand!(g, output_sym; namespace=ns)
NeuroDSL.patch_node!(g, patch_sym_check, corrupted_cache; namespace=ns)
NeuroDSL.demand!(g, output_sym; namespace=ns)
state_recompute_restore = NeuroDSL.capture_activations(g, ns)

all_match = all(isapprox(Array(state_cache_restore[k]), Array(state_recompute_restore[k]); atol=1f-6)
                for k in keys(state_cache_restore))
println(all_match ? "✅ Restauration par cache == restauration par recalcul (état exact)" :
                     "❌ ÉCHEC -- arrêt avant le benchmark de vitesse")
@assert all_match

# ── Benchmark : coût total d'un balayage complet des 8 couches ────────────
layer_sites = [Symbol(:layer_, i, :_out) for i in 1:n_layers]

function old_sweep!()
    for s in layer_sites
        NeuroDSL.patch_and_measure!(g, output_sym, s, clean_cache, corrupted_cache,
                                     clean_output, corrupted_output; namespace=ns)
    end
end

function new_sweep!()
    NeuroDSL.sweep_patch_sites!(g, output_sym, layer_sites, clean_cache, corrupted_cache,
                                 clean_output, corrupted_output; namespace=ns)
end

println("Warmup...")
old_sweep!(); new_sweep!()

sweep_times_old = Float64[]
sweep_times_new = Float64[]
for _ in 1:10
    t0 = time_ns(); old_sweep!(); push!(sweep_times_old, (time_ns() - t0) / 1e6)
    t0 = time_ns(); new_sweep!(); push!(sweep_times_new, (time_ns() - t0) / 1e6)
    GC.gc(false)
end

m_old, sd_old = trimmed_stats(sweep_times_old)
m_new, sd_new = trimmed_stats(sweep_times_new)
@printf "
Balayage complet (%d sites), restauration par recalcul : %.2f ms (±%.2f)
" n_layers m_old sd_old
@printf "Balayage complet (%d sites), restauration par cache    : %.2f ms (±%.2f)
" n_layers m_new sd_new
sweep_gain = (m_old - m_new) / m_old * 100
@printf "Gain (suppression du recalcul côté restauration)        : %.1f %%
" sweep_gain

✅ Restauration par cache == restauration par recalcul (état exact)
Warmup...

Balayage complet (8 sites), restauration par recalcul : 55.55 ms (±2.32)
Balayage complet (8 sites), restauration par cache    : 34.39 ms (±0.59)
Gain (suppression du recalcul côté restauration)        : 38.1 %


In [7]:
mkpath("../figures")

bar(["Recomputation
restore", "Cache replay
restore"], [m_old, m_new],
    yerr = [sd_old, sd_new],
    color = [:steelblue, :seagreen], legend = false,
    ylabel = "Total sweep cost (ms)",
    title = "Full sweep across $(n_layers) layers")
savefig("../figures/patching_sweep_cost.pdf")
println("✅ figures/patching_sweep_cost.pdf saved")

✅ figures/patching_sweep_cost.pdf saved


## Patching composable multi-nœuds

L'invalidation n'a jamais supposé qu'un seul nœud change à la fois : patcher plusieurs nœuds puis appeler `demand!` une seule fois calcule déjà l'union de leurs cônes en aval correctement, chaque nœud partagé n'étant recalculé qu'une fois. `patch_nodes!` expose cette propriété.

On choisit deux têtes d'attention **sœurs** de la même couche (`layer_1_mha_ao_h2` et `layer_1_mha_ao_h3`) : ni l'une ni l'autre n'est en amont de l'autre (toutes deux calculées depuis les mêmes Q/K/V, fusionnées ensuite par `hcat_heads`). C'est le cas où l'ordre d'application des patches ne doit structurellement jamais avoir d'importance -- contrairement à deux nœuds en relation ancêtre-descendant, où le patch le plus tardif dans l'ordre d'application l'emporterait aux points de recouvrement (comportement défini, mais dépendant de l'ordre).

In [8]:
site_a = Symbol(:layer_1_mha_ao_h2)
site_b = Symbol(:layer_1_mha_ao_h3)

# ── Correction : composition commutative (sites sœurs) ─────────────────────
NeuroDSL.patch_nodes!(g, [site_a, site_b], clean_cache; namespace=ns)
out_ab = copy(NeuroDSL.demand!(g, output_sym; namespace=ns))
NeuroDSL.restore_nodes_from_cache!(g, ns, corrupted_cache, [site_a, site_b])
NeuroDSL.demand!(g, output_sym; namespace=ns)

NeuroDSL.patch_node!(g, site_b, clean_cache; namespace=ns)
NeuroDSL.patch_node!(g, site_a, clean_cache; namespace=ns)
out_ba = copy(NeuroDSL.demand!(g, output_sym; namespace=ns))
commute_err = maximum(abs.(Array(out_ab) .- Array(out_ba)))
@printf "1) max|err| {A,B} vs {B,A} : %.6e  %s
" commute_err (commute_err < 1f-6 ? "✅" : "❌ ÉCHEC")
@assert commute_err < 1f-6
NeuroDSL.restore_nodes_from_cache!(g, ns, corrupted_cache, [site_a, site_b])
NeuroDSL.demand!(g, output_sym; namespace=ns)

# ── Récupération : individuelle vs combinée ─────────────────────────────────
r_a = NeuroDSL.patch_and_measure!(g, output_sym, site_a, clean_cache, corrupted_cache,
                                   clean_output, corrupted_output; namespace=ns).recovery
r_b = NeuroDSL.patch_and_measure!(g, output_sym, site_b, clean_cache, corrupted_cache,
                                   clean_output, corrupted_output; namespace=ns).recovery
NeuroDSL.patch_nodes!(g, [site_a, site_b], clean_cache; namespace=ns)
out_combined = NeuroDSL.demand!(g, output_sym; namespace=ns)
r_ab = NeuroDSL.recovery_metric(out_combined, clean_output, corrupted_output)
NeuroDSL.restore_nodes_from_cache!(g, ns, corrupted_cache, [site_a, site_b])
NeuroDSL.demand!(g, output_sym; namespace=ns)

@printf "
2) recovery(head 2 seule)      : %.4f
" r_a
@printf "   recovery(head 3 seule)      : %.4f
" r_b
@printf "   somme individuelle          : %.4f
" (r_a + r_b)
@printf "   recovery({head 2, head 3})  : %.4f
" r_ab

# ── Coût : cône combiné vs somme des cônes ──────────────────────────────────
cone_a = NeuroDSL._downstream_nodes(g, site_a, ns)
cone_b = NeuroDSL._downstream_nodes(g, site_b, ns)
cone_union = union(cone_a, cone_b)
@printf "
3) |cone(head 2)| = %d, |cone(head 3)| = %d, somme = %d
" length(cone_a) length(cone_b) (length(cone_a)+length(cone_b))
@printf "   |union|        = %d  (%.1f%% de la somme)
" length(cone_union) (length(cone_union)/(length(cone_a)+length(cone_b))*100)

1) max|err| {A,B} vs {B,A} : 0.000000e+00  ✅

2) recovery(head 2 seule)      : 0.0295
   recovery(head 3 seule)      : 0.0427
   somme individuelle          : 0.0723
   recovery({head 2, head 3})  : 0.0647

3) |cone(head 2)| = 493, |cone(head 3)| = 493, somme = 986
   |union|        = 494  (50.1% de la somme)


## Lecture des résultats

- **Coût** : décroît monotonement avec la profondeur -- patcher la couche 1 coûte ≈20% d'un forward complet (il faut recalculer 7 couches en aval), patcher la dernière couche coûte ≈0% (rien à recalculer après). Chez PyTorch/TransformerLens, les deux coûteraient exactement la même chose : un forward complet.
- **Récupération** : décroît elle aussi avec la profondeur -- restaurer le token corrompu tôt (couche 1) répare une plus grande partie de la sortie que le restaurer tard (couche 8), cohérent avec l'idée que l'information se mélange et se dilue au fil des couches d'attention.
- Les deux courbes ensemble racontent l'histoire : NeuroDSL permet de scanner tout le réseau à la recherche des couches causalement importantes, à un coût qui diminue avec la profondeur -- alors qu'un framework sans recalcul incrémental paierait le même prix (un forward complet) pour chaque couche testée, qu'elle soit causalement importante ou non.